In [1]:
import os

from src.consts import BASE_DATA_PATH

os.chdir("/home/rsoleyma/projects/twitter-stream-unpacker")

In [2]:
from src.db.models import DBPostIndexPost
from sqlalchemy import select
from src.db.db import init_pg_db

session = init_pg_db()()

In [3]:
session = init_pg_db()()
stmt = select(DBPostIndexPost.platform_id)
all_ids = [int(id) for id in session.execute(stmt).scalars().all()]

In [4]:
unique_ids = set(all_ids)
len(all_ids), len(unique_ids)

(18398471, 18239338)

In [8]:
from collections import Counter

counter = Counter(all_ids)
duplicates = {k: v for k, v in counter.items() if v > 1}
len(duplicates)

135425

In [7]:
total =  0
for k,v in duplicates.items():
    if v > 1:
        total += v-1
total

# {k:v for k,v in duplicates.items() if v > 2}

159133

In [9]:
from tqdm import tqdm
import more_itertools

seen_once_p_ids: list[str] = []
mark_for_deletion: list[int] = []

for batch in tqdm(more_itertools.batched(duplicates.keys(), 20000)):
    batch_ids = [str(id) for id in batch]
    stmt = select(DBPostIndexPost.id, DBPostIndexPost.platform_id).where(DBPostIndexPost.platform_id.in_(batch_ids))
    all_duplicate_rows = list(session.execute(stmt).all())
    print("query done")
    duplicates_sorted = sorted(all_duplicate_rows, key=lambda x: x[1])
    for id, platform_id in duplicates_sorted:
        if platform_id not in seen_once_p_ids:
            seen_once_p_ids.append(platform_id)
        else:
            mark_for_deletion.append(id)

7it [4:59:45, 2569.36s/it]


In [14]:
from pathlib import Path
import json
from src.consts import BASE_DATA_PATH
len(mark_for_deletion)
json.dump(mark_for_deletion, Path("/home/rsoleyma/projects/twitter-stream-unpacker/data/temp/unique_ids/mark_for_deletion.json()").open("w"), indent=2)


In [52]:
duplicates_sorted[0], duplicates_sorted[1]

((18726, '1477078079768076288'), (18728, '1477078079768076288'))

In [27]:
session.close()

In [62]:
stmt = select(DBPostIndexPost).where(DBPostIndexPost.platform_id == "1484185093031677952")
all_duplicate_rows2 = list(session.execute(stmt).scalars().all())
all_dupl_locs = [r.location_index for r in all_duplicate_rows2]

In [63]:
all_dupl_locs

[['2022-01', '20220120', '20220120/20220120152500.json.gz', 3011],
 ['2022-01', '20220120', '20220120/20220120152500.json.gz', 1079],
 ['2022-01', '20220120', '20220120/20220120152500.json.gz', 4937],
 ['2022-01', '20220120', '20220120/20220120152600.json.gz', 328],
 ['2022-01', '20220120', '20220120/20220120152500.json.gz', 3559],
 ['2022-01', '20220120', '20220120/20220120152600.json.gz', 2151],
 ['2022-01', '20220120', '20220120/20220120152600.json.gz', 1261]]